# ChatGPT-4o Response Collection
## Thesis Phase II — standalone collection notebook

Collects 30 responses (15 questions × EN + RU) from GPT-4o.
Output: `chatgpt_responses.json` saved to your evaluation folder.

**Runtime:** CPU is fine. No GPU needed.

In [ ]:
!pip install -q openai tqdm
print("✅ Done")

✅ Done


In [ ]:
import os, json, time
from pathlib import Path
from datetime import datetime
from tqdm import tqdm
print("Imports OK ✅")

Imports OK ✅


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
BASE     = Path("/content/drive/MyDrive/medical_protocols")
OUT_DIR  = BASE / "evaluation"
OUT_DIR.mkdir(parents=True, exist_ok=True)
QUESTIONS_JSON = BASE / "question_design_v1" / "final_questions.json"
print(f"OUT_DIR: {OUT_DIR}")

OUT_DIR: /content/drive/MyDrive/medical_protocols/evaluation


In [ ]:

OPENAI_API_KEY = ""   # ← paste here, e.g. "sk-proj-..."

if not OPENAI_API_KEY:
    try:
        from google.colab import userdata
        OPENAI_API_KEY = userdata.get("OPENAI_API_KEY") or ""
    except Exception:
        pass

print("OPENAI_API_KEY:", "✅ set" if OPENAI_API_KEY else "❌ MISSING — paste above")

OPENAI_API_KEY: ✅ set


In [ ]:
SYSTEM_PROMPT = (
    "You are a clinical assistant specialising in obstetrics and gynaecology. "
    "Answer the following question based on evidence-based clinical guidelines. "
    "Be specific and include: diagnostic criteria, medication names and dosages "
    "where relevant, gestational age thresholds, contraindications, and "
    "recommended follow-up intervals. "
    "Do not give general health advice — provide protocol-level clinical detail."
)
print("System prompt set ✅")

System prompt set ✅


In [ ]:
questions = json.loads(QUESTIONS_JSON.read_text(encoding="utf-8"))
print(f"Loaded {len(questions)} questions:")
for q in questions:
    print(f"  {q['id']} | {q['question_en'][:70]}...")

Loaded 15 questions:
  A1 | What are the diagnostic criteria for severe preeclampsia, and what blo...
  A2 | What fasting glucose and oral glucose tolerance test thresholds are us...
  A3 | What clinical and laboratory criteria define septic shock in obstetric...
  B1 | What is the recommended first-line antihypertensive therapy for severe...
  B2 | What empirical antibiotic regimen is recommended for obstetric sepsis,...
  B3 | What tocolytic agents are recommended for preterm labour, and what are...
  C1 | What are the stepwise surgical interventions recommended for postpartu...
  C2 | What are the indications for emergency hysterectomy in obstetric patie...
  C3 | What are the clinical signs and immediate management steps for suspect...
  D1 | What antenatal screening tests are recommended during the first trimes...
  D2 | What glucose monitoring targets are recommended for patients with gest...
  D3 | What risk factors identify pregnant women who should receive progester...
  E1 | 

In [ ]:

def query_chatgpt(question: str, system_prompt: str) -> str:
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)
    resp = client.chat.completions.create(
        model="gpt-4o",
        temperature=0,
        max_tokens=1024,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": question},
        ],
    )
    return resp.choices[0].message.content.strip()

print("Testing ChatGPT-4o...")
test = query_chatgpt("What is preeclampsia in one sentence?", SYSTEM_PROMPT)
print(f"Response: {test[:150]}...")
print("query_chatgpt() ready ✅")

Testing ChatGPT-4o...
Response: Preeclampsia is a pregnancy-specific hypertensive disorder characterized by new-onset hypertension (≥140/90 mmHg) and proteinuria (≥300 mg/24 hours) a...
query_chatgpt() ready ✅


In [ ]:
SAVE_PATH = OUT_DIR / "chatgpt_responses.json"

if SAVE_PATH.exists():
    collected = json.loads(SAVE_PATH.read_text(encoding="utf-8"))
    print(f"Resumed: {len(collected)} existing responses found.")
else:
    collected = []
    print("Starting fresh.")

done_keys = {(r["question_id"], r["language"]) for r in collected}
pending   = [(q, lang) for q in questions for lang in ["EN", "RU"]
             if (q["id"], lang) not in done_keys]
print(f"Pending: {len(pending)} / 30 queries")

for q, lang in tqdm(pending, desc="ChatGPT-4o"):
    q_text = q["question_en"] if lang == "EN" else q["question_ru"]
    try:
        response_text = query_chatgpt(q_text, SYSTEM_PROMPT)
    except Exception as e:
        response_text = f"ERROR: {e}"
        print(f"  ❌ {q['id']} {lang}: {e}")

    collected.append({
        "bot":             "ChatGPT-4o",
        "question_id":     q["id"],
        "stratum":         q.get("stratum", ""),
        "language":        lang,
        "question_text":   q_text,
        "response_text":   response_text,
        "target_protocol": q["target_protocol"],
        "timestamp":       datetime.utcnow().isoformat(),
    })
    SAVE_PATH.write_text(
        json.dumps(collected, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    time.sleep(1.0)

print(f"\n✅ Done! {len(collected)}/30 saved to {SAVE_PATH}")
errors = [r for r in collected if r["response_text"].startswith("ERROR:")]
print(f"Errors: {len(errors)} / {len(collected)}")
if errors:
    for r in errors:
        print(f"  ❌ {r['question_id']} {r['language']}: {r['response_text'][:100]}")
else:
    print("All responses clean ✅")

Starting fresh.
Pending: 30 / 30 queries


ChatGPT-4o:   0%|          | 0/30 [00:00<?, ?it/s]/tmp/ipykernel_506/2369093406.py:32: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp":       datetime.utcnow().isoformat(),
ChatGPT-4o: 100%|██████████| 30/30 [08:07<00:00, 16.24s/it]


✅ Done! 30/30 saved to /content/drive/MyDrive/medical_protocols/evaluation/chatgpt_responses.json
Errors: 0 / 30
All responses clean ✅
